In [ ]:
from riskmodels import RiskModelsClient
import os
os.getenv("RISKMODELS_API_KEY")
client = RiskModelsClient.from_env()
from Data import get_data, get_tickers, get_price_data, add_sector_subsector
import os
os.environ['GRPC_VERBOSITY'] = 'NONE'
os.environ['GLOG_minloglevel'] = '3'
import logging, warnings
logging.getLogger('hmmlearn').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=Warning, module='riskmodels')

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'JPM',
           'V', 'JNJ', 'WMT', 'PG', 'XOM', 'HD', 'BAC', 'KO', 'DIS', 'CSCO',
           'PFE', 'INTC']

In [ ]:
df = get_data(tickers=tickers, client=client, method=1, resample_window='W')
g = (df.dropna(subset=['sector','subsector'])
       .groupby(['sector','subsector'])['ticker'].nunique()
       .sort_values(ascending=False))
print(g)
print("usable groups (>=2 tickers):", int((g >= 2).sum()))
print("tickers in a usable group:", int(g[g >= 2].sum()))

In [ ]:
g = (df.dropna(subset=['sector','subsector'])
       .groupby(['sector','subsector'])['ticker']
       .apply(lambda s: sorted(s.unique())))

usable = g[g.apply(len) >= 2]

print("usable groups (>=2 tickers):", len(usable))
for (sec, sub), names in usable.items():
    print(f"  {sec} / {sub}: {names}")

used_tickers = sorted({t for names in usable for t in names})
print("\ntickers actually used:", used_tickers)
print("count:", len(used_tickers))

In [ ]:
from backtest_global import  backtest_global
from forward_beta import get_market_cap
df = get_market_cap(client, df, df['ticker'].unique())
bt = backtest_global(df, method=1, resample_window='W', book='mvo')


In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), height_ratios=[2, 1])
a1.plot(bt['date'], bt['port_growth_net'], label='port (net)', lw=2)
a1.plot(bt['date'], bt['spy_growth'], '--', label='SPY')
a1.plot(bt['date'], bt['ew_growth'], ':', label='EW')
a1.set_ylabel('Growth of $1'); a1.legend(); a1.grid(alpha=.3)
dd = lambda g: g / g.cummax() - 1
a2.fill_between(bt['date'], dd(bt['port_growth_net']), 0, alpha=.4)
a2.plot(bt['date'], dd(bt['spy_growth']), '--'); a2.set_ylabel('Drawdown'); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()